# 리포트 62 — CPI 를 늘리면 세 파형 모두 블라인드율이 내려간다

> ### 한 일
> **도플러 가드가 헤딩 축을 얼마나 지우는지를 CPI 격자에서 재고, 세 파형의 블라인드 헤딩 비율과 5G 가 치르는 배수를 같은 표에 실었다.**

### 결과
1. 도플러 축을 지우는 기구는 둘이다 — **표본화**: `guard_hz = g*PRF/M >= PRF/2  <=>  M <= 2g  <=>  T_cpi <= 2g/PRF [^1]` 로 가드가 접힘 축 전체를 덮는다(반송파·속도·거리 무관). **진폭**: 짧은 CPI 에서 가드가 도플러 진폭을 덮는다(파형 공통).
2. 5G 의 눈먼 헤딩 비율은 CPI 0.1 s [^2] 에서 0.636 [^3], CPI 0.2 s [^4] 에서 0.303 [^5] 로 내려간다.
3. WiFi 대비 배수는 CPI 0.1 [^6] ~ 2.0 s [^7] 5칸에서 11.0 [^8] ~ 19.0 [^9]배 로 남는다 — 이것이 이 대가를 구조로 만드는 첫 번째 사실이다.
4. 1.5빈 규약에서 LTE 도 CPI ≤ 0.024 s [^10] 에서 전 헤딩 블라인드가 된다 — 5G 만의 성질이 아니라 CPI 가 짧을 때의 성질이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 가드 규약 | 가드 반폭 = g빈 × PRF/M 이고 g 는 검출기 적용값 1.5빈과 선언값 2.5빈 **둘 다** 잰다 |
| 헤딩 격자 | 격자가 둘이다 — 발표된 앵커를 재현하는 72 점 [^11] 과 스윕이 쓰는 720 점 [^12] 이다. 속도는 5 m/s 로 고정한다 — `benchmark/cpi_guard_sweep.py` |
| 두 기구를 가른다 | 표본화가 만드는 접힘 축과 진폭이 만드는 덮임을 분리해 각각의 CPI 의존을 적는다 |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/sigma_sensitivity.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/cpi_guard_sweep.json` |
| 소요 | σ 격자 · 검지거리 4단계 · 검증 · 스윕을 합쳐 7.8 h [^13] (GPU 2 [^14]장). 이 빌더 자신은 CPU 수 초다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 50 «5G SSB 는 걷는 드론에서 접힌다»](50_doppler-fold.ipynb) | 물리 반복률이 만드는 접힘 |
| [편 52 «탭을 늘리면 환경이 정한 바닥에서 멈추고»](52_eca.ipynb) | 0-도플러 노치가 지우는 속도 |

---

## 도플러 축을 지우는 기구는 둘이다

5G 의 상시 기준신호(SSB)는 20 ms 주기라 PRF 50 Hz 를 준다. 그 축이 지워지는 경로가 둘이다.

**A. 표본화** — `guard_hz = g*PRF/M >= PRF/2  <=>  M <= 2g  <=>  T_cpi <= 2g/PRF [^1]` 로 가드가 접힘 축 전체를 덮는다(반송파·속도·거리 무관).

**B. 진폭** — 짧은 CPI 에서 가드가 도플러 진폭을 덮는다(파형 공통). 1.5빈 규약에서 LTE 도 CPI ≤ 0.024 s [^10] 에서 전 헤딩 블라인드가 된다.

## 기준 CPI 에서의 값 — 앵커 재현 격자

기준 CPI 는 0.1 s [^2] 다. 아래 표는 발표된 앵커를 그대로 재현하는 헤딩 격자 72 점 [^11] 위의 값이다.

| 모드 | PRF | 기준 CPI 의 M | 접힘 축 ± | 블라인드(1.5빈) | 블라인드(2.5빈) |
|---|---|---|---|---|---|
| WiFi | 1000 Hz [^15] | 100 [^16] | 500.0 Hz [^17] | 0.028 [^18] | 0.083 [^19] |
| LTE | 1000 Hz [^20] | 100 [^21] | 500.0 Hz [^22] | 0.139 [^23] | 0.250 [^24] |
| 5G | 50 Hz [^25] | 5 [^26] | 25.0 Hz [^27] | 0.639 [^28] | 1.000 [^29] |

5G 의 커버리지 0 은 선언가드 2.5빈 · CPI ≤ 0.10 s [^30] 에서 성립한다. 검출기가 적용하는 1.5빈 규약의 경계는 0.06 s [^31] 다.

두 수는 **표본화(A) 하나로** 정한 경계다. 진폭(B)까지 함께 관측한 경계는 각각 0.109 s [^32] · 0.070 s [^33] 이고, 아래 LTE 의 0.024 s [^10] 가 그 관측 잣대의 수다.

## CPI 를 늘리면 — 촘촘한 격자

세 파형 모두 블라인드율이 내려간다. 5G 가 치르는 **배수**는 5칸 전부에서 11.0 [^8] ~ 19.0 [^9]배 로 남는다 — 이것이 이 대가를 구조로 만드는 첫 번째 사실이다.

아래 표는 헤딩 격자 720 점 [^12] 위의 값이라, 같은 CPI 라도 앞의 앵커 재현 표와 값이 갈린다 — 격자가 촘촘하면 가드에 걸리는 헤딩 구간의 경계가 더 곱게 세어진다. 본문이 드는 수는 이 촘촘한 격자 쪽이다.

| CPI | WiFi | LTE | 5G | 5G/WiFi | 5G/LTE |
|---|---|---|---|---|---|
| 0.1 s [^2] | 0.053 [^34] | 0.158 [^35] | 0.636 [^36] | 12.1 [^37]배 | 4.0 [^38]배 |
| 0.2 s [^4] | 0.025 [^39] | 0.081 [^40] | 0.303 [^41] | 12.1 [^42]배 | 3.8 [^43]배 |
| 0.5 s [^44] | 0.008 [^45] | 0.031 [^46] | 0.119 [^47] | 14.3 [^48]배 | 3.9 [^49]배 |
| 1.0 s [^50] | 0.003 [^51] | 0.014 [^52] | 0.053 [^53] | 19.0 [^54]배 | 3.8 [^55]배 |
| 2.0 s [^56] | 0.003 [^57] | 0.008 [^58] | 0.031 [^59] | 11.0 [^60]배 | 3.7 [^61]배 |

## 두 번째 사실 — 접힘 비율은 CPI 와 무관하다

5G 의 alias 비율 0.861 [^62] 는 적분시간이 아니라 표본화율의 성질이라 CPI 와 무관한 상수이고, WiFi·LTE 는 0.000 [^63] 다.

세 번째 사실이 결정적이고, 그것은 [편 63 «모호속도는 표본화율의 성질이라 CPI 와 무관…»](63_cpi-residual.ipynb) 가 든다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| CPI 를 0.1 s 에서 1.0 s 까지 정본 solve 에 넣어 R90(CPI) 를 낸다 | 위 표의 커버리지 회복이 거리 축에서도 확정된다 | `benchmark/cpi_guard_sweep.py` → `src/experiment_freespace_range.py` |
| 헤딩 격자를 표적 기동 모형으로 바꿔 블라인드율을 다시 잰다 | 균일 헤딩 가정이 실제 비행에서 얼마나 낙관인지가 확정된다 | `benchmark/cpi_guard_sweep.py` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 63개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/cpi_guard_sweep.json` | `structural.formula` | guard_hz = g*PRF/M >= PRF/2  <=>  M <= 2g  <=>  T_cpi <… |
| [^2] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[0].T_cpi_s` | 0.1 |
| [^3] | `outputs/cpi_guard_sweep.json` | `verdict.artifact.blind_hard_same_cpi` | 0.6361 |
| [^4] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[1].T_cpi_s` | 0.2 |
| [^5] | `outputs/cpi_guard_sweep.json` | `verdict.artifact.blind_hard_at_200ms` | 0.3028 |
| [^6] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[*].T_cpi_s → 최소` | (여러 칸) |
| [^7] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[*].T_cpi_s → 최대` | (여러 칸) |
| [^8] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[*].ratio_G1_over_W1 → 5칸 최소` | (여러 칸) |
| [^9] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[*].ratio_G1_over_W1 → 5칸 최대` | (여러 칸) |
| [^10] | `outputs/cpi_guard_sweep.json` | `structural.two_mechanisms.observed.L1.hard.T_max_total_blind_s` | 0.02436 |
| [^11] | `outputs/cpi_guard_sweep.json` | `meta.psi_n_published` | 72 |
| [^12] | `outputs/cpi_guard_sweep.json` | `meta.psi_n_fine` | 720 |
| [^13] | `outputs/report05_derived.json` | `runtime.total_h` | 7.773 |
| [^14] | `outputs/report13_freespace.json` | `meta.gpus` | 2 |
| [^15] | `outputs/cpi_guard_sweep.json` | `waveform_facts.W1.prf_hz` | 1000 |
| [^16] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.W1.M` | 100 |
| [^17] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.W1.fold_half_hz` | 500 |
| [^18] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.W1.blind_hard` | 0.02778 |
| [^19] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.W1.blind_declared` | 0.08333 |
| [^20] | `outputs/cpi_guard_sweep.json` | `waveform_facts.L1.prf_hz` | 1000 |
| [^21] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.L1.M` | 100 |
| [^22] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.L1.fold_half_hz` | 500 |
| [^23] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.L1.blind_hard` | 0.1389 |
| [^24] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.L1.blind_declared` | 0.25 |
| [^25] | `outputs/cpi_guard_sweep.json` | `waveform_facts.G1.prf_hz` | 50 |
| [^26] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.G1.M` | 5 |
| [^27] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.G1.fold_half_hz` | 25 |
| [^28] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.G1.blind_hard` | 0.6389 |
| [^29] | `outputs/cpi_guard_sweep.json` | `anchor.reproduction.G1.blind_declared` | 1 |
| [^30] | `outputs/cpi_guard_sweep.json` | `structural.by_mode.G1.T_max_total_blind_declared_s` | 0.1 |
| [^31] | `outputs/cpi_guard_sweep.json` | `structural.by_mode.G1.T_max_total_blind_hard_s` | 0.06 |
| [^32] | `outputs/cpi_guard_sweep.json` | `structural.two_mechanisms.observed.G1.declared.T_max_total_blind_s` | 0.1086 |
| [^33] | `outputs/cpi_guard_sweep.json` | `structural.two_mechanisms.observed.G1.hard.T_max_total_blind_s` | 0.06959 |
| [^34] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[0].blind_hard_W1` | 0.05278 |
| [^35] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[0].blind_hard_L1` | 0.1583 |
| [^36] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[0].blind_hard_G1` | 0.6361 |
| [^37] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[0].ratio_G1_over_W1` | 12.05 |
| [^38] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[0].ratio_G1_over_L1` | 4.018 |
| [^39] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[1].blind_hard_W1` | 0.025 |
| [^40] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[1].blind_hard_L1` | 0.08056 |
| [^41] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[1].blind_hard_G1` | 0.3028 |
| [^42] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[1].ratio_G1_over_W1` | 12.11 |
| [^43] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[1].ratio_G1_over_L1` | 3.759 |
| [^44] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[2].T_cpi_s` | 0.5 |
| [^45] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[2].blind_hard_W1` | 0.008333 |
| [^46] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[2].blind_hard_L1` | 0.03056 |
| [^47] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[2].blind_hard_G1` | 0.1194 |
| [^48] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[2].ratio_G1_over_W1` | 14.33 |
| [^49] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[2].ratio_G1_over_L1` | 3.909 |
| [^50] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[3].T_cpi_s` | 1 |
| [^51] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[3].blind_hard_W1` | 0.002778 |
| [^52] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[3].blind_hard_L1` | 0.01389 |
| [^53] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[3].blind_hard_G1` | 0.05278 |
| [^54] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[3].ratio_G1_over_W1` | 19 |
| [^55] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[3].ratio_G1_over_L1` | 3.8 |
| [^56] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[4].T_cpi_s` | 2 |
| [^57] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[4].blind_hard_W1` | 0.002778 |
| [^58] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[4].blind_hard_L1` | 0.008333 |
| [^59] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[4].blind_hard_G1` | 0.03056 |
| [^60] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[4].ratio_G1_over_W1` | 11 |
| [^61] | `outputs/cpi_guard_sweep.json` | `equal_cpi_penalty[4].ratio_G1_over_L1` | 3.667 |
| [^62] | `outputs/cpi_guard_sweep.json` | `verdict.structural.s2_alias_floor.alias_frac_G1` | 0.8611 |
| [^63] | `outputs/cpi_guard_sweep.json` | `verdict.structural.s2_alias_floor.alias_frac_W1` | 0 |